# Fintech Pipeline Data Processing

In [13]:
import pandas as pd
import json
import os

# Create a directory for processed data if it doesn't exist
output_dir = '01_data/processed'
os.makedirs(output_dir, exist_ok=True)

# --- Create dummy JSON for demonstration if it doesn't exist ---

# Dummy api_response_sample.json
api_response_data = {
    "transactions": [
        {
            "id": "T1001",
            "details": {
                "amount": 150.75,
                "currency": "USD",
                "status": "success"
            },
            "customer": {
                "id": "C001",
                "name": "Alice"
            },
            "items": [
                {"item_id": "I001", "quantity": 1},
                {"item_id": "I002", "quantity": 2}
            ]
        },
        {
            "id": "T1002",
            "details": {
                "amount": 200.00,
                "currency": "EUR",
                "status": "pending"
            },
            "customer": {
                "id": "C002",
                "name": "Bob"
            },
            "items": []
        }
    ],
    "metadata": {
        "timestamp": "2023-10-26T10:00:00Z",
        "source": "api"
    }
}
with open('api_response_sample.json', 'w') as f:
    json.dump(api_response_data, f, indent=4)

print("Dummy api_response_sample.json created (assuming ledger.csv and gateway.csv are uploaded).")

Dummy api_response_sample.json created (assuming ledger.csv and gateway.csv are uploaded).


## 1. Reconciliation: Load and Merge Data

In [14]:
# Load ledger.csv and gateway.csv
df_ledger = pd.read_csv('ledger.csv')
df_gateway = pd.read_csv('gateway.csv')

print("Ledger Data:")
display(df_ledger.head())
print("\nGateway Data:")
display(df_gateway.head())

# Perform a full outer merge with indicator=True
# This helps identify records present only in left, only in right, or in both.
reconciliation_df = pd.merge(df_ledger, df_gateway, on='id', how='outer', suffixes=('_ledger', '_gateway'), indicator=True)

print("\nReconciliation Results (first 5 rows):")
display(reconciliation_df.head())

# Find missing records
missing_in_gateway = reconciliation_df[reconciliation_df['_merge'] == 'left_only']
missing_in_ledger = reconciliation_df[reconciliation_df['_merge'] == 'right_only']

print("\nRecords in Ledger but not in Gateway (missing in Gateway):")
display(missing_in_gateway[['id', 'amount_ledger', 'status_ledger']])

print("\nRecords in Gateway but not in Ledger (missing in Ledger):")
display(missing_in_ledger[['id', 'amount_gateway', 'status_gateway']])

Ledger Data:


,id,amount,status,timestamp
0,A001,100.50,completed,2023-01-01
1,A002,200.00,pending,2023-01-01
2,A003,50.25,completed,2023-01-02
3,A004,75.00,completed,2023-01-02
4,A005,120.00,failed,2023-01-03



Gateway Data:


,id,amount,status,gateway_ref
0,A001,100.50,completed,G001
1,A002,205.00,approved,G002
2,A003,50.25,completed,G003
3,A005,120.00,declined,G005
4,A007,400.00,processed,G007



Reconciliation Results (first 5 rows):


,id,amount_ledger,status_ledger,timestamp,amount_gateway,status_gateway,gateway_ref,_merge
0,A001,100.50,completed,2023-01-01,100.50,completed,G001,both
1,A002,200.00,pending,2023-01-01,205.00,approved,G002,both
2,A003,50.25,completed,2023-01-02,50.25,completed,G003,both
3,A004,75.00,completed,2023-01-02,NaN,NaN,NaN,left_only
4,A005,120.00,failed,2023-01-03,120.00,declined,G005,both



Records in Ledger but not in Gateway (missing in Gateway):


,id,amount_ledger,status_ledger
3,A004,75.0,completed
5,A006,300.0,completed



Records in Gateway but not in Ledger (missing in Ledger):


,id,amount_gateway,status_gateway
6,A007,400.0,processed
7,A008,150.0,completed


### 1.1 Data Quality Checks: Duplicates and Nulls

In [15]:
print("--- Ledger Data Quality ---")
print("Ledger Duplicates:", df_ledger.duplicated().sum())
print("Ledger Nulls:\n", df_ledger.isnull().sum())

print("\n--- Gateway Data Quality ---")
print("Gateway Duplicates:", df_gateway.duplicated().sum())
print("Gateway Nulls:\n", df_gateway.isnull().sum())

--- Ledger Data Quality ---
Ledger Duplicates: 0
Ledger Nulls:
 id           0
amount       0
status       0
timestamp    0
dtype: int64

--- Gateway Data Quality ---
Gateway Duplicates: 0
Gateway Nulls:
 id             0
amount         0
status         0
gateway_ref    0
dtype: int64


## 2. Mismatches: Compare Columns for Matching IDs

In [16]:
# Filter for records present in both dataframes
matched_records = reconciliation_df[reconciliation_df['_merge'] == 'both'].copy()

# Compare 'amount' and 'status' columns where IDs match
mismatched_amounts = matched_records[matched_records['amount_ledger'] != matched_records['amount_gateway']]
mismatched_statuses = matched_records[matched_records['status_ledger'] != matched_records['status_gateway']]

print("Mismatched Amounts:")
display(mismatched_amounts[['id', 'amount_ledger', 'amount_gateway']])

print("\nMismatched Statuses:")
display(mismatched_statuses[['id', 'status_ledger', 'status_gateway']])

# Combine all mismatches for a comprehensive view
# You might want to define a custom comparison logic based on business rules
mismatched_records = matched_records[
    (matched_records['amount_ledger'] != matched_records['amount_gateway']) |
    (matched_records['status_ledger'] != matched_records['status_gateway'])
]

print("\nAll Mismatched Records (where ID matches but amount or status differs):")
display(mismatched_records[['id', 'amount_ledger', 'amount_gateway', 'status_ledger', 'status_gateway']])

Mismatched Amounts:


,id,amount_ledger,amount_gateway
1,A002,200.0,205.0



Mismatched Statuses:


,id,status_ledger,status_gateway
1,A002,pending,approved
4,A005,failed,declined



All Mismatched Records (where ID matches but amount or status differs):


,id,amount_ledger,amount_gateway,status_ledger,status_gateway
1,A002,200.0,205.0,pending,approved
4,A005,120.0,120.0,failed,declined


## 3. JSON: Flatten `api_response_sample.json`

In [17]:
# Load the JSON file
with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)

# --- Step 1: Normalize transactions (excluding the 'items' list for now) ---
df_transactions_base = pd.json_normalize(
    api_data['transactions'],
    sep='_',
    # Exclude 'items' from initial flatten as it's a list of dicts that needs exploding
    # The 'items' column will remain a list of dictionaries at this stage
    # We'll handle 'items' in a separate step.
)

# --- Step 2: Incorporate top-level metadata into each transaction row ---
metadata_df = pd.json_normalize(api_data['metadata'])
# Duplicate metadata values for each transaction row
for col in metadata_df.columns:
    df_transactions_base[f'metadata_{col}'] = metadata_df[col].iloc[0]

# --- Step 3: Flatten the nested 'items' list ---
# Explode the 'items' column, creating a new row for each item in a transaction
df_exploded_items = df_transactions_base.explode('items')

# Normalize the 'items' column itself. Handle cases where 'items' might be NaN (e.g., empty list in original JSON)
# pd.json_normalize handles lists of dicts. If 'items' contains non-dict values (like NaN after explode), filter them.
df_normalized_items = df_exploded_items['items'].apply(lambda x: pd.Series(x) if isinstance(x, dict) else pd.Series({}))

# Add prefix to item columns to avoid conflicts with transaction columns
df_normalized_items = df_normalized_items.add_prefix('item_')

# Reset index of df_exploded_items to align correctly for concatenation
# And drop the original 'items' column as it's now flattened
df_exploded_items_final = df_exploded_items.reset_index(drop=True).drop(columns=['items'])

# Reset the index of df_normalized_items to ensure it aligns with df_exploded_items_final
df_normalized_items = df_normalized_items.reset_index(drop=True)

# Combine the exploded and normalized items with the rest of the transaction data
df_api_normalized = pd.concat([df_exploded_items_final, df_normalized_items], axis=1)

# --- Step 4: Clean column names ---
# Example: Convert all column names to snake_case and remove potential remaining '.'
df_api_normalized.columns = [col.replace('.', '_').lower() for col in df_api_normalized.columns]

# --- Step 5: Convert date/time fields ---
if 'metadata_timestamp' in df_api_normalized.columns:
    df_api_normalized['metadata_timestamp'] = pd.to_datetime(df_api_normalized['metadata_timestamp'])

print("Normalized API Data (first 5 rows):")
display(df_api_normalized.head())

# --- Step 6: Save the normalized output ---
api_normalized_file_path = os.path.join(output_dir, 'api_normalized.csv')
df_api_normalized.to_csv(api_normalized_file_path, index=False)
print(f"Saved normalized API data to {api_normalized_file_path}")

Normalized API Data (first 5 rows):


,id,details_amount,details_currency,details_status,customer_id,customer_name,metadata_timestamp,metadata_source,item_item_id,item_quantity
0,T1001,150.75,USD,success,C001,Alice,2023-10-26 10:00:00+00:00,api,I001,1.0
1,T1001,150.75,USD,success,C001,Alice,2023-10-26 10:00:00+00:00,api,I002,2.0
2,T1002,200.00,EUR,pending,C002,Bob,2023-10-26 10:00:00+00:00,api,NaN,NaN


Saved normalized API data to 01_data/processed/api_normalized.csv


## 4. Export: Save Processed Data to CSVs

In [18]:
# The user requested saving all required CSVs into `01_data/processed/`.

# Ensure the output directory exists (created at the beginning of the notebook)

# Save reconciliation report
reconciliation_df.to_csv(os.path.join(output_dir, 'reconciliation_report.csv'), index=False)
print(f"Saved reconciliation_report.csv to {output_dir}")

# Save missing records in gateway
missing_in_gateway.to_csv(os.path.join(output_dir, 'missing_in_gateway.csv'), index=False)
print(f"Saved missing_in_gateway.csv to {output_dir}")

# Save missing records in ledger
missing_in_ledger.to_csv(os.path.join(output_dir, 'missing_in_ledger.csv'), index=False)
print(f"Saved missing_in_ledger.csv to {output_dir}")

# Save amount mismatches
mismatched_amounts.to_csv(os.path.join(output_dir, 'amount_mismatches.csv'), index=False)
print(f"Saved amount_mismatches.csv to {output_dir}")

# Save status mismatches
mismatched_statuses.to_csv(os.path.join(output_dir, 'status_mismatches.csv'), index=False)
print(f"Saved status_mismatches.csv to {output_dir}")

# Save all mismatched records (this was originally 'mismatched_transactions.csv', keeping for comprehensive view)
mismatched_records.to_csv(os.path.join(output_dir, 'mismatched_transactions.csv'), index=False)
print(f"Saved mismatched_transactions.csv to {output_dir}")

# df_flattened_transactions is now superseded by df_api_normalized, which is saved in cell e4d6ac70
# print(f"Saved flattened_api_transactions.csv to {output_dir}") # Removed as it's now handled by api_normalized.csv

print(f"\nAll specified CSVs saved to '{output_dir}/'.")

Saved reconciliation_report.csv to 01_data/processed
Saved missing_in_gateway.csv to 01_data/processed
Saved missing_in_ledger.csv to 01_data/processed
Saved amount_mismatches.csv to 01_data/processed
Saved status_mismatches.csv to 01_data/processed
Saved mismatched_transactions.csv to 01_data/processed

All specified CSVs saved to '01_data/processed/'.


## 5. Metrics: Create `summary_metrics.json`

The request is to *manually* create `04_python/summary_metrics.json` using counts from your Python code and provide instructions on how to do this in Google Colab.

Here's how you can manually create or update this JSON file in Google Colab:

1.  **Run the Python code above** to get your counts (e.g., number of `missing_in_gateway` records, `mismatched_amounts`, etc.).
2.  **Open the Colab Files pane**: Click the folder icon on the left sidebar (Files).
3.  **Navigate to the directory**: If you want to create it in `04_python/`, you might need to create that directory first. Right-click in the files pane -> `New folder` -> type `04_python`.
4.  **Create a new file**: Right-click in the `04_python` folder -> `New file` -> type `summary_metrics.json`.
5.  **Edit the file**: Double-click `summary_metrics.json`. An editor will open in the main pane.
6.  **Populate with data**: Manually enter the JSON structure, referencing the counts obtained from your Python code. For example:

    ```json
    {
        "reconciliation_metrics": {
            "total_ledger_records": 6,
            "total_gateway_records": 6,
            "records_only_in_ledger": 2,
            "records_only_in_gateway": 2,
            "records_in_both": 4
        },
        "mismatch_metrics": {
            "amount_mismatches": 1,
            "status_mismatches": 2
        },
        "flattened_api_metrics": {
            "total_transactions_flattened": 2,
            "unique_customers": 2
        }
    }
    ```

7.  **Save**: Colab usually auto-saves, but you can also press `Ctrl + S` (or `Cmd + S` on Mac).

Alternatively, you can use Python code to generate this JSON programmatically after computing the metrics:

In [19]:
# Calculate summary metrics from the dataframes
summary_metrics = {
    "reconciliation_metrics": {
        "total_ledger_records": len(df_ledger),
        "total_gateway_records": len(df_gateway),
        "records_only_in_ledger": len(missing_in_gateway),
        "records_only_in_gateway": len(missing_in_ledger),
        "records_in_both": len(matched_records)
    },
    "mismatch_metrics": {
        "amount_mismatches": len(mismatched_amounts),
        "status_mismatches": len(mismatched_statuses),
        "total_mismatched_records": len(mismatched_records)
    },
    "flattened_api_metrics": {
        "total_transactions_flattened": len(df_api_normalized), # Updated to use df_api_normalized
        "unique_customers": df_api_normalized['customer_id'].nunique() if 'customer_id' in df_api_normalized.columns else 0 # Updated to use df_api_normalized
    }
}

# Create the directory if it doesn't exist
metrics_dir = '04_python'
os.makedirs(metrics_dir, exist_ok=True)

# Save the summary metrics to a JSON file programmatically
metrics_file_path = os.path.join(metrics_dir, 'summary_metrics.json')
with open(metrics_file_path, 'w') as f:
    json.dump(summary_metrics, f, indent=4)

print(f"Summary metrics saved to {metrics_file_path}")

# Display the generated JSON content
print("\nGenerated summary_metrics.json content:")
print(json.dumps(summary_metrics, indent=4))

Summary metrics saved to 04_python/summary_metrics.json

Generated summary_metrics.json content:
{
    "reconciliation_metrics": {
        "total_ledger_records": 6,
        "total_gateway_records": 6,
        "records_only_in_ledger": 2,
        "records_only_in_gateway": 2,
        "records_in_both": 4
    },
    "mismatch_metrics": {
        "amount_mismatches": 1,
        "status_mismatches": 2,
        "total_mismatched_records": 2
    },
    "flattened_api_metrics": {
        "total_transactions_flattened": 3,
        "unique_customers": 2
    }
}
